In [ ]:
# https://kikaben.com/yolov5-transfer-learning-dogs-cats/
# https://sandipanweb.wordpress.com/2022/01/17/custom-object-detection-with-transfer-learning-with-pre-trained-yolo-v4-model/

%pip install firebase_admin
%pip install Pillow
%pip install numpy

In [ ]:
from PIL import Image, ImageDraw
from os import listdir

print(listdir())

def show_bbox(image_path):
    # convert image path to label path
    label_path = image_path#.replace('/images/', '/darknet/')
    label_path = label_path.replace('.jpg', '.txt')
    print(label_path)
    # Open the image and create ImageDraw object for drawing
    image = Image.open(image_path)
    draw = ImageDraw.Draw(image)

    with open(label_path, 'r') as f:
        for line in f.readlines():
            # Split the line into five values
            label, x, y, w, h = line.split(' ')

            # Convert string into float
            x = float(x)
            y = float(y)
            w = float(w)
            h = float(h)

            # Convert center position, width, height into
            # top-left and bottom-right coordinates
            W, H = image.size
            x1 = (x - w/2) * W
            y1 = (y - h/2) * H
            x2 = (x + w/2) * W
            y2 = (y + h/2) * H

            # Draw the bounding box with red lines
            draw.rectangle((x1, y1, x2, y2),
                           outline=(255, 0, 0), # Red in RGB
                           width=5)             # Line width
    image.show()
    

    
show_bbox('yolo/test/001-Sketch.jpg')

In [ ]:
import os

import firebase_admin
from firebase_admin import credentials

# Point these at your own Firebase project (see the README).
cred = credentials.Certificate(os.environ["FIREBASE_CREDENTIALS"])  # path to a service account JSON key
firebase_admin.initialize_app(cred, {
    'databaseURL': os.environ["FIREBASE_DATABASE_URL"],
    'storageBucket': os.environ["FIREBASE_STORAGE_BUCKET"]
})

In [ ]:
import os, shutil

folder = 'yolo/data'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))

In [ ]:
import re

def extract_file(url):

    # Define the pattern to extract the desired substring
    pattern = r'^.*%2F(.*?.png)'

    # Use regex to find the matching substring
    match = re.search(pattern, url)

    if match:
        extracted_string = match.group(1)
        return extracted_string
    else:
        print("No match found.")
        
print(file)

In [ ]:
from firebase_admin import db, storage
# import ast

from PIL import Image


bucket = storage.bucket()
print(bucket)
# for key in bucket.keys():
#     print(f"Bucket key: {key}")

ref = db.reference("/")
all_dates = ref.get()
print(list(all_dates.keys())[0])

classifications = {'button': 0, 'image': 1, 'label': 2, 'navigationBar': 3, 'tabBar': 4, 'textField': 5, 'switchButton': 6}

# classifications = ["button": ]

classification_count = {}
drawings_count = 0

def extract_drawings(date):
    global drawings_count
    all_drawings = ref.child(date).get()
    for drawing in all_drawings.keys():
        child = ref.child(date).child(drawing).get()
        print(child["image"])
        drawings_count = drawings_count + 1
        file_name = extract_file(child["image"])
        blob = bucket.blob(date + '/' + drawing + '/' + file_name)
        print(blob)
        print(blob.exists())
        download_file_name = folder + "/" + file_name
        print(download_file_name)
        blob.download_to_filename(download_file_name)
        im = Image.open(download_file_name)

        new_image = Image.new("RGBA", im.size, "WHITE") # Create a white rgba background
        new_image.paste(im, (0, 0), im)              # Paste the image on the background. Go to the links given below for details.
        new_image.convert('RGB').save(download_file_name.replace('.png', '.jpg'), "JPEG")  # Save as JPEG
        
        image_width, image_height = im.size

        bounding_boxes_file_name = download_file_name.replace('.png', '.txt')

        with open(bounding_boxes_file_name, 'w+') as file:

            elements = ref.child(date).child(drawing).child("elements").get()
            print(f"DRAWING: {drawing}")
            print()
            print("Iterating elements:")
            for element in elements:
                if element == None:
                    continue
                
#                 print(element['type'])
                if element['type'] == "unknown":
                    continue
                if element['type'] not in classifications:
                    print(f"We're not looking for '{element['type']}' yet")
                    continue

                classification_count[element['type']] = classification_count.get(element['type'], 0) + 1

                # Remove the outer curly braces to make it a valid Python literal
                inner_data_string = element['box'].strip('{}')
                # Split the string into individual point-size pairs
                pairs = inner_data_string.split('}, {')
                parsed_data = [tuple(map(int, pair.strip('{}').split(', '))) for pair in pairs]
                origin = parsed_data[0]
                size = parsed_data[1]
                
                if element['type'] == "navigationBar":
                    nav_height = origin[1] + size[1]
                    centre_x = 0.5 #(origin[0] + (size[0] / 2)) / image_width
                    centre_y = (0.0 + (nav_height / 2)) / image_height
                    relative_width = 1.0 #size[0] / image_width
                    relative_height = nav_height / image_height
                    print(f"{download_file_name} has NavBar")
                    if relative_height > 0.25:
                        # Catch any bad data!
                        print(f"Found bad nav bar data! {date} / {drawing} / {file_name}")
                        print(f"Original nav bounds: {pairs}")
                        print(centre_x)
                        print(centre_y)
                        print(relative_width)
                        print(relative_height)
                        continue
                elif element['type'] == "tabBar":
                    tab_height = image_height - origin[1]
                    centre_x = 0.5 #(origin[0] + (size[0] / 2)) / image_width
                    centre_y = (origin[1] + (tab_height / 2)) / image_height
                    relative_width = 1.0 #size[0] / image_width
                    relative_height = tab_height / image_height
                    print(f"{download_file_name} has TabBar")
                    if relative_height > 0.25:
                        # Catch any bad data!
                        print(f"Found bad tab bar data! {date} / {drawing} / {file_name}")
                        print(f"Original tab bounds: {pairs}")
                        print(centre_x)
                        print(centre_y)
                        print(relative_width)
                        print(relative_height)
                        continue
                    
                else:
                    centre_x = (origin[0] + (size[0] / 2)) / image_width
                    centre_y = (origin[1] + (size[1] / 2)) / image_height
                    relative_width = size[0] / image_width
                    relative_height = size[1] / image_height
#                 print(centre_x)
#                 print(centre_y)
#                 print(relative_width)
#                 print(relative_height)

                output_data = [classifications[element['type']], centre_x, centre_y, relative_width, relative_height]

                file.write(' '.join(str(value) for value in output_data))
                file.write('\n')

for date in all_dates.keys():
    if date == "stats":
        # ignore our stats counter
        continue
    extract_drawings(date)

for element in classification_count:
    print(f"{element} -> {classification_count[element]}")

print("Classification count:")
print(classification_count)
print(f"Total drawings: {drawings_count}")
        
print("DONE")

In [ ]:
show_bbox('yolo/data/51689851-876D-4516-B004-847FA8C5C4A7.jpg')

In [ ]:
import os

# Create a folder structure for YOLOv5 training
if not os.path.exists('yolo/data/training'):
    for folder in ['images', 'labels']:
        for split in ['train', 'val', 'test']:
            os.makedirs(f'yolo/data/training/{folder}/{split}')

In [ ]:
import glob

def get_filenames(folder):
    filenames = set()
    
    for path in glob.glob(os.path.join(folder, '*.jpg')):
        # Extract the filename
        filename = os.path.split(path)[-1]        
        filenames.add(filename)

    return filenames

In [ ]:
import numpy as np

sketch_images = get_filenames('yolo/data')
print(len(sketch_images))

sketch_images = np.array(list(sketch_images))

# Use the same random seed for reproducability
np.random.seed(42)
np.random.shuffle(sketch_images)
print(sketch_images)

In [ ]:
import shutil
import math

def split_dataset(image_names, train_size, val_size):
    for i, image_name in enumerate(image_names):
        
#         if image_name == "99DF3059-62B0-4EEB-B46D-4C57BF21C3AC.jpg":
#             continue
#         if image_name == "53343E27-4B36-4923-9A79-5F4ACE074164.jpg":
#             continue
        
        # Label filename
        label_name = image_name.replace('.jpg', '.txt')
        
        # Split into train, val, or test
        if i < train_size:
            split = 'train'
        elif i < train_size + val_size:
            split = 'val'
        else:
            split = 'test'
        
        # Source paths
        source_image_path = f'yolo/data/{image_name}'
        source_label_path = f'yolo/data/{label_name}'

        # Destination paths
        target_image_folder = f'yolo/data/training/images/{split}'
        target_label_folder = f'yolo/data/training/labels/{split}'

        # Copy files
        shutil.copy(source_image_path, target_image_folder)
        shutil.copy(source_label_path, target_label_folder)

print(len(sketch_images))
total = len(sketch_images)
train_size = math.floor(total * 0.9) #was .75
val_size = math.floor(total * 0.05) #was .15
# val_size = total - 2 - train_size
# TODO: We should leave space for test!! Any remainder goes in test by default
split_dataset(sketch_images, train_size=train_size, val_size=val_size)

We've created a yaml file specifying the data directories
Then we're running:

```
python yolov5/train.py --data ui_components.yaml --weights yolov5s.pt --epochs 100 --batch 4 --freeze 10
```
to freeze the backbone of the model (this uses the small model - yolov5s - which has 10 layers)

This command also does the training.

We've also tried 

```
python yolov5/train.py --data ui_components.yaml --weights yolov5s.pt --epochs 300 --img 1280 --batch 4 --freeze 10  
```
which allows a larger resolution (640 is the default) and more epochs. The results weren't massively better than the prior setting

In [ ]:
%pwd
%cd yolo/yolo
!python yolov5/train.py --data ui_components.yaml --weights yolov5s.pt --epochs 300 --batch 4 --freeze 10

To try out the detection we run 
```
python yolov5/detect.py --weights yolov5/runs/train/exp3/weights/best.pt --conf 0.4 --img 1280 --source ../data/training/images/test
```